# TFT – vergleichbar zu Prophet & Chronos

Wertet **TFT** nach derselben Logik aus wie Prophet/Chronos:

- **Split:** 3 Jahre Training / 1 Jahr Test (aus den `prepared`-CSVs).
- **Rollierende 72-h-Fenster** über das Testjahr, nach **Lead-Zeit gebucketet** (8/24/48/72 h) — wie Prophet 12.3.
- **Metriken:** MAE, RMSE, **MASE** (Skala = Seasonal-Naive-Fehler des Trainings, m=24), **MAPE**.
- **Regressoren:** Wetter (+ Feiertag + Zeit-Features) als `futr_exog` → Perfect Prognosis (wie Prophet).

Kernel **Python (tft)** wählen, dann Run All. Bei CUDA-OOM `WINDOWS_BATCH`/`HIDDEN_SIZE` weiter senken (siehe Config).

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os, gc
import numpy as np
import pandas as pd
import torch
import holidays
from pathlib import Path
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MAE as NF_MAE

torch.set_float32_matmul_precision('high')

STATION       = 'Aotizhongxin'
FREQ          = 'h'
H_TFT         = 72
INPUT_SIZE    = 168
MAX_STEPS     = 1000
BATCH_SIZE    = 32
HIDDEN_SIZE   = 64          # kleiner = weniger GPU-Speicher (Default 128)
WINDOWS_BATCH = 128         # WICHTIG gegen CUDA-OOM (Default 1024); bei OOM weiter senken (64/32)
HORIZONTE     = [8, 24, 48, 72]
DATA_DIR      = Path('../data/prepared')
REGRESSOREN   = ['TEMP', 'DEWP', 'PRES', 'WSPM', 'RAIN', 'wd_sin', 'wd_cos']

print('GPU:', torch.cuda.is_available())

GPU: True


In [2]:
def lade(variante, station=STATION):
    tr = pd.read_csv(DATA_DIR / variante / f'prophet_train_{station}.csv', parse_dates=['ds'])
    te = pd.read_csv(DATA_DIR / variante / f'prophet_test_{station}.csv',  parse_dates=['ds'])
    return tr, te

def add_feiertag(df):
    cn = holidays.country_holidays('CN', years=range(2013, 2018))
    d = df.copy()
    d['feiertag'] = d['ds'].dt.date.astype('O').map(lambda t: 1 if t in cn else 0)
    return d

def add_time_features(df):
    d = df.copy()
    d['hour_sin'] = np.sin(2*np.pi*d['ds'].dt.hour/24); d['hour_cos'] = np.cos(2*np.pi*d['ds'].dt.hour/24)
    d['day_sin']  = np.sin(2*np.pi*d['ds'].dt.dayofyear/365.25); d['day_cos'] = np.cos(2*np.pi*d['ds'].dt.dayofyear/365.25)
    return d

def regularize(df, spalten):
    df = df.copy(); df['ds'] = pd.to_datetime(df['ds'])
    d = df.set_index('ds').sort_index()
    d = d.reindex(pd.date_range(d.index.min(), d.index.max(), freq=FREQ))
    for c in spalten:
        d[c] = d[c].ffill().bfill() if c == 'feiertag' else d[c].interpolate(limit_direction='both')
    d.index.name = 'ds'; return d.reset_index()

def zu_nf_format(df, regressoren, feiertag_flag, uid=STATION):
    d = add_time_features(df)
    if feiertag_flag: d = add_feiertag(d)
    d.insert(0, 'unique_id', uid)
    cols = ['unique_id','ds','y'] + regressoren + ['hour_sin','hour_cos','day_sin','day_cos'] + (['feiertag'] if feiertag_flag else [])
    return d[cols]

def mae(y, yh):  return float(np.mean(np.abs(np.asarray(y,float)-np.asarray(yh,float))))
def rmse(y, yh): return float(np.sqrt(np.mean((np.asarray(y,float)-np.asarray(yh,float))**2)))
def mape(y, yh):
    y=np.asarray(y,float); yh=np.asarray(yh,float); m=y>0
    return float(np.mean(np.abs((y[m]-yh[m])/y[m]))*100) if m.any() else np.nan
def mase_skala(train_y, mp=24):
    a=np.asarray(train_y,float); return float(np.nanmean(np.abs(a[mp:]-a[:-mp])))

In [3]:
# --- Einzelstation ---
def tft_eval(variante, regressoren, feiertag, log, name):
    tr, te = lade(variante)
    skala = mase_skala(tr['y'])
    exog = regressoren + ['hour_sin','hour_cos','day_sin','day_cos'] + (['feiertag'] if feiertag else [])
    gesamt = regularize(pd.concat([tr, te], ignore_index=True), spalten=['y']+regressoren)
    voll = zu_nf_format(gesamt, regressoren, feiertag)
    if log: voll = voll.assign(y=np.log1p(voll['y']))
    n_windows = len(te) // H_TFT
    model = TFT(h=H_TFT, input_size=INPUT_SIZE, hist_exog_list=exog, futr_exog_list=exog,
                loss=NF_MAE(), max_steps=MAX_STEPS, batch_size=BATCH_SIZE,
                hidden_size=HIDDEN_SIZE, windows_batch_size=WINDOWS_BATCH,
                inference_windows_batch_size=WINDOWS_BATCH,
                accelerator='gpu', devices=1, precision='16-mixed', enable_progress_bar=True)
    nf = NeuralForecast(models=[model], freq=FREQ)
    cv = nf.cross_validation(df=voll, n_windows=n_windows, step_size=H_TFT)
    if log:
        cv['TFT'] = np.expm1(cv['TFT']); cv['y'] = np.expm1(cv['y'])
    cv['yhat'] = cv['TFT'].clip(lower=0)
    cv['lead'] = ((cv['ds'] - cv['cutoff']).dt.total_seconds() // 3600).astype(int)
    rows = []
    for b in HORIZONTE:
        w = cv[cv['lead'] <= b]
        rows.append({'Modell': name, 'Horizont': f'{b} h',
                     'MAE': mae(w['y'], w['yhat']), 'RMSE': rmse(w['y'], w['yhat']),
                     'MASE': mae(w['y'], w['yhat'])/skala, 'MAPE %': mape(w['y'], w['yhat'])})
    del model, nf, cv; gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(rows)

konfigs = {
    'TFT: Basis (univariat)':    dict(variante='basis',     regressoren=[],          feiertag=False, log=False),
    'TFT: Behandelt + FT + Reg': dict(variante='behandelt', regressoren=REGRESSOREN, feiertag=True,  log=False),
}
alle = []
for name, cfg in konfigs.items():
    print('== Trainiere', name, '=='); torch.cuda.empty_cache()
    tab = tft_eval(name=name, **cfg)
    print(tab.round(3).to_string(index=False)); print(); alle.append(tab)

ergebnis_tft = pd.concat(alle, ignore_index=True)[['Modell','Horizont','MAE','RMSE','MASE','MAPE %']].round(3)
print('=== Gesamt ==='); print(ergebnis_tft.to_string(index=False))
ergebnis_tft.to_csv('../data/ergebnis_tft.csv', index=False)
print('gespeichert -> ../data/ergebnis_tft.csv')

Seed set to 1
Using 16bit Automatic Mixed Precision (AMP)


== Trainiere TFT: Basis (univariat) ==


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                    | Type                     | Params
---------------------------------------------------------------------
0 | loss                    | MAE                      | 0     
1 | padder_train            | ConstantPad1d            | 0     
2 | scaler                  | TemporalNorm             | 0     
3 | embedding               | TFTEmbedding             | 1.2 K 
4 | temporal_encoder        | TemporalCovariateEncoder | 370 K 
5 | temporal_fusion_decoder | TemporalFusionDecoder    | 64.8 K
6 | output_adapter          | Linear                   | 65    
---------------------------------------------------------------------
436 K     Trainable params
0         Non-trainable params
436 K     Total params
1.747     Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1000` reached.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

                   Modell Horizont    MAE   RMSE  MASE  MAPE %
TFT: Behandelt + FT + Reg      8 h 27.608 41.631 0.461  66.804
TFT: Behandelt + FT + Reg     24 h 28.975 44.469 0.483  66.672
TFT: Behandelt + FT + Reg     48 h 33.085 54.441 0.552  71.071
TFT: Behandelt + FT + Reg     72 h 34.086 54.170 0.569  75.821

=== Gesamt ===
                   Modell Horizont    MAE   RMSE  MASE  MAPE %
   TFT: Basis (univariat)      8 h 24.590 40.237 0.415  69.324
   TFT: Basis (univariat)     24 h 36.384 57.580 0.614 119.887
   TFT: Basis (univariat)     48 h 49.203 77.473 0.831 154.768
   TFT: Basis (univariat)     72 h 56.314 86.875 0.951 192.992
TFT: Behandelt + FT + Reg      8 h 27.608 41.631 0.461  66.804
TFT: Behandelt + FT + Reg     24 h 28.975 44.469 0.483  66.672
TFT: Behandelt + FT + Reg     48 h 33.085 54.441 0.552  71.071
TFT: Behandelt + FT + Reg     72 h 34.086 54.170 0.569  75.821
gespeichert -> ../data/ergebnis_tft.csv


## Variante: alle 12 Stationen (globales TFT)

Ein **globales TFT** über alle 12 Stationen (je Station eine `unique_id`), **ein** Fit. Auswertung wie Prophet 12.3: rollierend, Lead-Zeit-Buckets, MASE je Station und über die 12 Stationen gemittelt. Schreibt `../data/ergebnis_tft_allstations.csv`.

> Speicher-intensiver als Einzelstation → bei CUDA-OOM zuerst `WINDOWS_BATCH` senken (128 → 64 → 32) und ggf. `HIDDEN_SIZE` 64 → 32. Prophet nutzt 12 getrennte Modelle; hier ein gemeinsames (üblich für TFT).

In [4]:
stationen = sorted(p.name.replace('prophet_train_', '').replace('.csv', '')
                   for p in (DATA_DIR / 'basis').glob('prophet_train_*.csv'))
print(len(stationen), 'Stationen:', stationen)

def baue_multi(variante, regressoren, feiertag):
    teile, skalen = [], {}
    for st in stationen:
        tr, te = lade(variante, st)
        skalen[st] = mase_skala(tr['y'])
        g = regularize(pd.concat([tr, te], ignore_index=True), spalten=['y'] + regressoren)
        teile.append(zu_nf_format(g, regressoren, feiertag, uid=st))
    return pd.concat(teile, ignore_index=True), skalen

def tft_eval_all(variante, regressoren, feiertag, log, name):
    multi, skalen = baue_multi(variante, regressoren, feiertag)
    if log: multi = multi.assign(y=np.log1p(multi['y']))
    exog = regressoren + ['hour_sin','hour_cos','day_sin','day_cos'] + (['feiertag'] if feiertag else [])
    tr0, te0 = lade(variante)
    test_h = int((te0['ds'].max() - te0['ds'].min()) / pd.Timedelta(hours=1)) + 1
    n_windows = test_h // H_TFT
    model = TFT(h=H_TFT, input_size=INPUT_SIZE, hist_exog_list=exog, futr_exog_list=exog,
                loss=NF_MAE(), max_steps=MAX_STEPS, batch_size=BATCH_SIZE,
                hidden_size=HIDDEN_SIZE, windows_batch_size=WINDOWS_BATCH,
                inference_windows_batch_size=WINDOWS_BATCH,
                accelerator='gpu', devices=1, precision='16-mixed', enable_progress_bar=True)
    nf = NeuralForecast(models=[model], freq=FREQ)
    cv = nf.cross_validation(df=multi, n_windows=n_windows, step_size=H_TFT)
    if log:
        cv['TFT'] = np.expm1(cv['TFT']); cv['y'] = np.expm1(cv['y'])
    cv['yhat'] = cv['TFT'].clip(lower=0)
    cv['lead'] = ((cv['ds'] - cv['cutoff']).dt.total_seconds() // 3600).astype(int)
    rows = []
    for b in HORIZONTE:
        w = cv[cv['lead'] <= b]
        per = [{'MAE': mae(g['y'], g['yhat']), 'RMSE': rmse(g['y'], g['yhat']),
                'MASE': mae(g['y'], g['yhat']) / skalen[st], 'MAPE %': mape(g['y'], g['yhat'])}
               for st, g in w.groupby('unique_id')]
        m = pd.DataFrame(per).mean()
        rows.append({'Modell': name, 'Horizont': f'{b} h', 'MAE': m['MAE'], 'RMSE': m['RMSE'],
                     'MASE': m['MASE'], 'MAPE %': m['MAPE %']})
    del model, nf, cv; gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(rows)

konfigs_all = {
    'TFT 12 Stat.: Basis (univariat)':    dict(variante='basis',     regressoren=[],          feiertag=False, log=False),
    'TFT 12 Stat.: Behandelt + FT + Reg': dict(variante='behandelt', regressoren=REGRESSOREN, feiertag=True,  log=False),
}
alle_all = []
for name, cfg in konfigs_all.items():
    print('== Globales TFT ueber 12 Stationen:', name, '=='); torch.cuda.empty_cache()
    tab = tft_eval_all(name=name, **cfg)
    print(tab.round(3).to_string(index=False)); print(); alle_all.append(tab)

ergebnis_tft_all = pd.concat(alle_all, ignore_index=True)[['Modell','Horizont','MAE','RMSE','MASE','MAPE %']].round(3)
print('=== Gesamt (12 Stationen, Mittel) ==='); print(ergebnis_tft_all.to_string(index=False))
ergebnis_tft_all.to_csv('../data/ergebnis_tft_allstations.csv', index=False)
print('gespeichert -> ../data/ergebnis_tft_allstations.csv')

12 Stationen: ['Aotizhongxin', 'Changping', 'Dingling', 'Dongsi', 'Guanyuan', 'Gucheng', 'Huairou', 'Nongzhanguan', 'Shunyi', 'Tiantan', 'Wanliu', 'Wanshouxigong']
== Globales TFT ueber 12 Stationen: TFT 12 Stat.: Basis (univariat) ==


Seed set to 1
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                    | Type                     | Params
---------------------------------------------------------------------
0 | loss                    | MAE                      | 0     
1 | padder_train            | ConstantPad1d            | 0     
2 | scaler                  | TemporalNorm             | 0     
3 | embedding               | TFTEmbedding             | 1.2 K 
4 | temporal_encoder        | TemporalCovariateEncoder | 370 K 
5 | temporal_fusion_decoder | TemporalFusionDecoder    | 64.8 K
6 | output_adapter          | Linear                   | 65    
---------------------------------------------------------------------
436 K     Trainable params
0         Non-trainable params
436 K     Total params
1.747     Tota

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1000` reached.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

                            Modell Horizont    MAE   RMSE  MASE  MAPE %
TFT 12 Stat.: Behandelt + FT + Reg      8 h 26.209 41.339 0.444  65.527
TFT 12 Stat.: Behandelt + FT + Reg     24 h 28.188 45.525 0.480  68.253
TFT 12 Stat.: Behandelt + FT + Reg     48 h 33.006 56.713 0.562  72.512
TFT 12 Stat.: Behandelt + FT + Reg     72 h 34.847 58.187 0.594  79.621

=== Gesamt (12 Stationen, Mittel) ===
                            Modell Horizont    MAE   RMSE  MASE  MAPE %
   TFT 12 Stat.: Basis (univariat)      8 h 22.870 39.375 0.389  83.074
   TFT 12 Stat.: Basis (univariat)     24 h 36.187 58.905 0.617 131.297
   TFT 12 Stat.: Basis (univariat)     48 h 48.310 77.586 0.824 172.717
   TFT 12 Stat.: Basis (univariat)     72 h 52.208 79.684 0.891 199.616
TFT 12 Stat.: Behandelt + FT + Reg      8 h 26.209 41.339 0.444  65.527
TFT 12 Stat.: Behandelt + FT + Reg     24 h 28.188 45.525 0.480  68.253
TFT 12 Stat.: Behandelt + FT + Reg     48 h 33.006 56.713 0.562  72.512
TFT 12 Stat.: Behandelt +

### Hinweise

- **Bei CUDA out of memory:** `WINDOWS_BATCH` senken (128 → 64 → 32), dann `HIDDEN_SIZE` (64 → 32), zur Not `BATCH_SIZE`. Nach einem OOM-Absturz **Kernel neu starten** (der GPU-Speicher ist sonst blockiert).
- **Vergleichbarkeit:** entspricht Prophet 12.3 (rollierend, Lead-Zeit-Buckets) — der repräsentative Weg.
- **Perfect Prognosis:** `futr_exog` nutzt echtes Zukunftswetter, wie Prophets Regressor-Modelle.
- **Training:** `MAX_STEPS` steuert die Länge (nicht Epochen); für belastbare Zahlen 1000–3000.